# ChestX FIX-first iterative feature improvement

This notebook is a cleaned-up replacement for the earlier search notebook.

## What changed

Your earlier notebook already had a useful **candidate-bank generator** and a **local search loop**, but the optimization target was mixed:
- proposals were **accepted by a reward**
- the best solution was **tracked by FIX**
- `faith` was computed but not actually used in the reward

That makes it hard to tell whether **FIX itself** can drive iterative improvement.

This notebook separates the problem into three experiments:

1. **FIX-only search baseline**
   - accept by FIX
   - track best by FIX
   - use diversity only as a tiebreaker

2. **Stronger search**
   - random restarts
   - beam-style expansion
   - better diagnostics

3. **Actual RL-like policy learning**
   - a small REINFORCE policy over edit actions
   - reward is **delta FIX**
   - this is a real learned policy, unlike pure local search

## Core idea

FIX is best used as an **evaluator** over a structured proposal space.

That means:
- FIX can guide iterative improvement
- but not from truly unconstrained masks from scratch

So the best practical recipe is:
- build a strong candidate bank
- run a FIX-consistent search first
- only then test whether a learned policy helps

In [ ]:
# Optional installs for a fresh Colab runtime
# %pip install -q exlib captum matplotlib scikit-image pandas tqdm

In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from skimage import measure, morphology
from tqdm.auto import tqdm

from exlib.datasets.chestx import ChestXDataset, ChestXFixScore, ChestXPathologyModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

PATHOLOGY_NAMES = ChestXDataset.pathology_names
STRUCTURE_NAMES = ChestXDataset.structure_names

## Load the ChestX data and model

This uses the same official `exlib` ChestX pieces as your original notebook.

In [ ]:
setattr(torch.backends, "cudnn", torch.backends.cudnn)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

test_ds = ChestXDataset(split="test")
print("test size:", len(test_ds))

task_model = ChestXPathologyModel().to(device)
try:
    task_model = ChestXPathologyModel.from_pretrained("BrachioLab/chestx_pathols").to(device)
    print("Loaded pretrained ChestXPathologyModel.")
except Exception as e:
    print("Could not load pretrained weights, using default init instead.")
    print("Reason:", repr(e))

task_model.eval()

fix_scorer = ChestXFixScore().to(device)
print("FIX scorer ready.")

## Helpers

This section includes:
- image and tensor utilities
- corrected Grad-CAM hook
- candidate-bank construction
- FIX scoring
- visualization helpers

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_numpy01(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().float().numpy()
    x = np.asarray(x, dtype=np.float32).squeeze()
    if x.size == 0:
        return x
    lo, hi = float(x.min()), float(x.max())
    if hi - lo < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return (x - lo) / (hi - lo)


def get_logits(model, x):
    out = model(x)
    if hasattr(out, "logits"):
        out = out.logits
    if isinstance(out, dict):
        out = out.get("logits", list(out.values())[0])
    return out


def choose_feature_layer(model) -> str:
    names = [name for name, _ in model.named_modules()]
    preferred = [
        name for name in names
        if any(k in name for k in ["denseblock4", "norm5", "transition3"])
    ]
    if preferred:
        return preferred[-1]
    for name in reversed(names):
        if name:
            return name
    raise RuntimeError("Could not find a feature layer.")


def disable_inplace_relu(module):
    for child in module.children():
        if isinstance(child, torch.nn.ReLU):
            child.inplace = False
        disable_inplace_relu(child)


disable_inplace_relu(task_model)
feature_layer = choose_feature_layer(task_model)
print("feature_layer:", feature_layer)

In [ ]:
def gradcam_heatmaps(model, x, layer_name: str, target_indices):
    acts = {}
    module = dict(model.named_modules())[layer_name]

    def fwd_hook(_m, _inp, out):
        if isinstance(out, (tuple, list)):
            out = out[0]
        acts["value"] = out
        out.retain_grad()

    h = module.register_forward_hook(fwd_hook)
    try:
        x = x.to(next(model.parameters()).device).clone().detach().requires_grad_(True)
        model.zero_grad(set_to_none=True)
        logits = get_logits(model, x)
        score = logits[0, target_indices].sum()
        score.backward()

        if "value" not in acts:
            raise RuntimeError(f"Forward hook did not capture activations for layer '{layer_name}'.")

        a = acts["value"]
        g = a.grad

        if g is None:
            raise RuntimeError(
                f"No gradient captured for layer '{layer_name}'. "
                "Try a different feature layer or make sure this layer participates in the score computation."
            )
    finally:
        h.remove()

    if a.dim() == 3:
        a = a.unsqueeze(0)
    if g.dim() == 3:
        g = g.unsqueeze(0)

    weights = g.mean(dim=(-1, -2), keepdim=True)
    cam = torch.relu((weights * a).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]

    channel_scores = (
        weights.flatten() * a.flatten(2).mean(-1).flatten()
    ).detach().cpu().numpy()

    up = F.interpolate(a, size=x.shape[-2:], mode="bilinear", align_corners=False)[0]

    return to_numpy01(cam), up.detach().cpu().numpy(), channel_scores

In [ ]:
def connected_components(mask, min_area: int):
    labels = measure.label(mask)
    comps = []
    for region in sorted(measure.regionprops(labels), key=lambda r: r.area, reverse=True):
        if region.area < min_area:
            continue
        comp = labels == region.label
        comp = morphology.remove_small_holes(comp, area_threshold=max(64, min_area))
        comp = morphology.remove_small_objects(comp, min_size=min_area)
        if comp.sum() >= min_area:
            comps.append(comp.astype(bool))
    return comps


def extract_masks_from_map(map_2d, image01, quantiles, min_area):
    masks = []
    grad = np.abs(np.gradient(image01)[0]) + np.abs(np.gradient(image01)[1])
    grad = to_numpy01(grad)

    support = morphology.binary_dilation(
        grad > np.quantile(grad, 0.35),
        morphology.disk(2),
    )

    fmap = to_numpy01(map_2d)
    for q in quantiles:
        base = fmap >= np.quantile(fmap, q)
        base = morphology.binary_opening(base, morphology.disk(1))
        base = morphology.binary_closing(base, morphology.disk(3))
        base = base & morphology.binary_dilation(support, morphology.disk(6))
        masks.extend(connected_components(base, min_area=min_area))

    return masks


def mask_iou(a, b) -> float:
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / max(float(union), 1.0)


def dedup_masks(masks, iou_thr=0.65):
    kept = []
    for mask in masks:
        if mask.sum() == 0:
            continue
        if all(mask_iou(mask, prev) < iou_thr for prev in kept):
            kept.append(mask.astype(bool))
    return kept


def pick_targets(logits, gt_pathols, topk=3):
    gt_idx = torch.where(gt_pathols[0].detach().cpu() > 0.5)[0].tolist()
    if gt_idx:
        return gt_idx
    probs = torch.sigmoid(logits[0].detach().cpu())
    return torch.topk(probs, k=min(topk, probs.numel())).indices.tolist()

In [ ]:
def build_candidate_bank(
    model,
    image_tensor,
    gt_pathols,
    layer_name,
    max_candidates=12,
    min_area=120,
    n_top_channels=6,
):
    image01 = to_numpy01(image_tensor[0, 0])
    logits = get_logits(model, image_tensor.to(next(model.parameters()).device))
    targets = pick_targets(logits, gt_pathols)
    cam, up_channels, channel_scores = gradcam_heatmaps(model, image_tensor, layer_name, targets)

    order = np.argsort(channel_scores)[::-1]
    masks = []
    provenance = []

    for rank in range(min(n_top_channels, len(order))):
        cidx = int(order[rank])
        channel_map = up_channels[cidx]
        for comp in extract_masks_from_map(channel_map, image01, quantiles=[0.80, 0.85, 0.90], min_area=min_area):
            masks.append(comp)
            provenance.append(f"channel_{cidx}")

    for comp in extract_masks_from_map(cam, image01, quantiles=[0.72, 0.80, 0.88], min_area=min_area):
        masks.append(comp)
        provenance.append("gradcam")

    masks = dedup_masks(masks)[:max_candidates]
    provenance = provenance[:len(masks)]

    return {
        "targets": targets,
        "logits": logits.detach().cpu(),
        "image01": image01,
        "cam": cam,
        "groups": masks,
        "provenance": provenance,
        "channel_scores": channel_scores,
    }

In [ ]:
def groups_to_tensor(groups, device):
    if len(groups) == 0:
        raise ValueError("groups must be non-empty")
    arr = np.stack([g.astype(np.float32) for g in groups], axis=0)
    return torch.from_numpy(arr).unsqueeze(0).to(device)


@torch.no_grad()
def score_groups_fix(fix_scorer, groups, structs_tensor, device):
    pred = groups_to_tensor(groups, device=device)
    true = structs_tensor.unsqueeze(0).to(device)
    score = fix_scorer(groups_pred=pred, groups_true=true)
    return float(score.flatten()[0].item())


@torch.no_grad()
def pathology_drop(model, image_tensor, groups, targets):
    device = next(model.parameters()).device
    logits_full = get_logits(model, image_tensor.to(device))[0]
    base = torch.sigmoid(logits_full[targets]).mean().item()

    union = np.logical_or.reduce(groups) if groups else np.zeros(image_tensor.shape[-2:], dtype=bool)
    keep = torch.from_numpy((~union).astype(np.float32)).to(device)[None, None]
    logits_occ = get_logits(model, image_tensor.to(device) * keep)[0]
    occ = torch.sigmoid(logits_occ[targets]).mean().item()
    return float(base - occ)


def diversity_score(groups):
    if len(groups) <= 1:
        return 1.0
    overlaps = []
    for i in range(len(groups)):
        for j in range(i + 1, len(groups)):
            overlaps.append(mask_iou(groups[i], groups[j]))
    return 1.0 - float(np.mean(overlaps))

In [ ]:
def metrics_fix_only(fix_scorer, model, image_tensor, structs_tensor, groups, targets):
    fix = score_groups_fix(fix_scorer, groups, structs_tensor, next(model.parameters()).device)
    div = diversity_score(groups)
    faith = pathology_drop(model, image_tensor, groups, targets)
    return {
        "fix": fix,
        "div": div,
        "faith": faith,
        "num_groups": len(groups),
    }


def better_by_fix_then_div(a: Dict[str, float], b: Dict[str, float], eps: float = 1e-8) -> bool:
    if a["fix"] > b["fix"] + eps:
        return True
    if abs(a["fix"] - b["fix"]) <= eps and a["div"] > b["div"] + eps:
        return True
    return False

In [ ]:
def mutate_mask(mask, image01, rng):
    op = rng.choice(["dilate", "erode", "open", "close", "fill", "largest", "snap"])
    k = int(rng.integers(1, 5))
    selem = morphology.disk(k)
    out = mask.copy()

    if op == "dilate":
        out = morphology.binary_dilation(out, selem)
    elif op == "erode":
        out = morphology.binary_erosion(out, selem)
    elif op == "open":
        out = morphology.binary_opening(out, selem)
    elif op == "close":
        out = morphology.binary_closing(out, selem)
    elif op == "fill":
        out = morphology.remove_small_holes(out, area_threshold=256)
    elif op == "largest":
        comps = connected_components(out, min_area=1)
        if comps:
            out = comps[0]
    elif op == "snap":
        grad = np.abs(np.gradient(image01)[0]) + np.abs(np.gradient(image01)[1])
        grad = to_numpy01(grad)
        outer = morphology.binary_dilation(out, morphology.disk(5))
        inner = morphology.binary_erosion(out, morphology.disk(2))
        band = np.logical_and(outer, ~inner)
        thresh = np.quantile(grad[band], 0.6) if band.sum() else 0.5
        out = np.logical_or(inner, np.logical_and(outer, grad >= thresh))

    out = morphology.remove_small_objects(out.astype(bool), min_size=80)
    out = morphology.remove_small_holes(out.astype(bool), area_threshold=128)
    return out.astype(bool)


def sanitize_groups(groups, candidate_bank, min_area=80):
    proposal = [g for g in dedup_masks(groups) if g.sum() >= min_area]
    if not proposal and len(candidate_bank) > 0:
        proposal = [candidate_bank[0].copy()]
    return proposal


def apply_action(proposal, action, candidate_bank, image01, rng, max_groups=8):
    proposal = [g.copy() for g in proposal]

    if action == "modify" and proposal:
        idx = int(rng.integers(0, len(proposal)))
        proposal[idx] = mutate_mask(proposal[idx], image01, rng)

    elif action == "add" and len(proposal) < max_groups and candidate_bank:
        proposal.append(candidate_bank[int(rng.integers(0, len(candidate_bank)))].copy())

    elif action == "delete" and len(proposal) > 1:
        proposal.pop(int(rng.integers(0, len(proposal))))

    elif action == "replace" and proposal and candidate_bank:
        idx = int(rng.integers(0, len(proposal)))
        proposal[idx] = candidate_bank[int(rng.integers(0, len(candidate_bank)))].copy()

    elif action == "merge" and len(proposal) > 1:
        i, j = sorted(rng.choice(len(proposal), size=2, replace=False).tolist())
        proposal[i] = np.logical_or(proposal[i], proposal[j])
        proposal.pop(j)

    return sanitize_groups(proposal, candidate_bank)

## Visualization

In [ ]:
def show_overlay(ax, image01, mask, title="", color=(1.0, 0.15, 0.15), alpha=0.35):
    ax.imshow(image01, cmap="gray", vmin=0, vmax=1)
    rgba = np.zeros((*mask.shape, 4), dtype=np.float32)
    rgba[..., :3] = np.array(color, dtype=np.float32)
    rgba[..., 3] = alpha * mask.astype(np.float32)
    ax.imshow(rgba)
    for contour in measure.find_contours(mask.astype(float), 0.5):
        ax.plot(contour[:, 1], contour[:, 0], color="white", lw=1.2)
    ax.set_title(title, fontsize=10)
    ax.axis("off")


def plot_group_grid(image01, groups, title):
    cols = min(4, max(1, len(groups)))
    rows = int(math.ceil(max(1, len(groups)) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)
    colors = [
        (1.0, 0.15, 0.15),
        (0.1, 0.8, 1.0),
        (0.35, 1.0, 0.35),
        (1.0, 0.8, 0.1),
        (0.9, 0.3, 1.0),
        (1.0, 0.5, 0.2),
    ]
    for i, ax in enumerate(axes):
        if i < len(groups):
            show_overlay(ax, image01, groups[i], title=f"group {i}", color=colors[i % len(colors)])
        else:
            ax.axis("off")
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()


def best_structure_match(group, structs):
    ious = [mask_iou(group, structs[i].astype(bool)) for i in range(structs.shape[0])]
    best = int(np.argmax(ious))
    return best, float(ious[best])


def plot_structure_matches(image01, groups, structs):
    rows = min(len(groups), 6)
    fig, axes = plt.subplots(rows, 2, figsize=(8, 4 * rows))
    axes = np.atleast_2d(axes)
    for i in range(rows):
        best_idx, best_iou = best_structure_match(groups[i], structs)
        show_overlay(axes[i, 0], image01, groups[i], title=f"discovered group {i}")
        show_overlay(
            axes[i, 1],
            image01,
            structs[best_idx].astype(bool),
            title=f"best structure: {STRUCTURE_NAMES[best_idx]}\nIoU={best_iou:.3f}",
            color=(0.1, 0.8, 1.0),
        )
    fig.tight_layout()
    plt.show()

## Experiment 1: FIX-only search baseline

This is the diagnostic baseline you asked for.

It answers:

> If we optimize **only FIX**, can the search move FIX at all?

Rules:
- accept by FIX
- track best by FIX
- use diversity only as a tiebreaker

In [ ]:
def improve_groups_fix_only(
    fix_scorer,
    model,
    image_tensor,
    structs_tensor,
    init_groups,
    candidate_bank,
    targets,
    num_steps=120,
    max_groups=8,
    seed=0,
):
    rng = np.random.default_rng(seed)
    image01 = to_numpy01(image_tensor[0, 0])

    current = [g.copy() for g in init_groups]
    current_metrics = metrics_fix_only(fix_scorer, model, image_tensor, structs_tensor, current, targets)

    best = [g.copy() for g in current]
    best_metrics = dict(current_metrics)

    history = [dict(step=0, accepted=True, action="init", **current_metrics)]

    for step in range(1, num_steps + 1):
        action = rng.choice(["modify", "add", "delete", "replace", "merge"])
        proposal = apply_action(current, action, candidate_bank, image01, rng, max_groups=max_groups)
        proposal_metrics = metrics_fix_only(fix_scorer, model, image_tensor, structs_tensor, proposal, targets)

        accept = better_by_fix_then_div(proposal_metrics, current_metrics)
        if accept:
            current = [g.copy() for g in proposal]
            current_metrics = dict(proposal_metrics)

        if better_by_fix_then_div(current_metrics, best_metrics):
            best = [g.copy() for g in current]
            best_metrics = dict(current_metrics)

        history.append(dict(step=step, accepted=accept, action=action, **current_metrics))

    return best, history, best_metrics

## Experiment 2: stronger search

Two upgrades:
- **random restarts** to escape local optima
- **beam search** to keep several strong candidates alive at once

In practice, these often matter more than “making it RL”.

In [ ]:
def ranked_singletons(fix_scorer, model, image_tensor, structs_tensor, candidate_bank, targets):
    rows = []
    for i, g in enumerate(candidate_bank):
        m = metrics_fix_only(fix_scorer, model, image_tensor, structs_tensor, [g], targets)
        rows.append((i, m["fix"], m["div"], m["faith"]))
    rows.sort(key=lambda x: (x[1], x[2]), reverse=True)
    return rows


def seed_init_groups(candidate_bank, singleton_rows, init_groups=4):
    idxs = [r[0] for r in singleton_rows[:min(init_groups, len(singleton_rows))]]
    return [candidate_bank[i].copy() for i in idxs]

In [ ]:
def run_fix_search_with_restarts(
    fix_scorer,
    model,
    image_tensor,
    structs_tensor,
    candidate_bank,
    targets,
    n_restarts=8,
    num_steps=100,
    init_groups=4,
    max_groups=8,
    seed=0,
):
    rng = np.random.default_rng(seed)
    singleton_rows = ranked_singletons(
        fix_scorer, model, image_tensor, structs_tensor, candidate_bank, targets
    )

    all_runs = []
    best_groups = None
    best_metrics = None

    for r in range(n_restarts):
        if r == 0:
            start_groups = seed_init_groups(candidate_bank, singleton_rows, init_groups=init_groups)
        else:
            order = rng.permutation(len(candidate_bank)).tolist()
            chosen = order[:min(init_groups, len(order))]
            start_groups = [candidate_bank[i].copy() for i in chosen]

        groups, history, metrics = improve_groups_fix_only(
            fix_scorer=fix_scorer,
            model=model,
            image_tensor=image_tensor,
            structs_tensor=structs_tensor,
            init_groups=start_groups,
            candidate_bank=candidate_bank,
            targets=targets,
            num_steps=num_steps,
            max_groups=max_groups,
            seed=int(rng.integers(0, 10_000_000)),
        )
        all_runs.append({"restart": r, "groups": groups, "history": history, "metrics": metrics})

        if best_metrics is None or better_by_fix_then_div(metrics, best_metrics):
            best_groups = [g.copy() for g in groups]
            best_metrics = dict(metrics)

    return best_groups, all_runs, best_metrics, singleton_rows

In [ ]:
def beam_search_fix(
    fix_scorer,
    model,
    image_tensor,
    structs_tensor,
    candidate_bank,
    targets,
    beam_width=4,
    expansions_per_state=4,
    num_rounds=12,
    max_groups=8,
    seed=0,
):
    rng = np.random.default_rng(seed)
    image01 = to_numpy01(image_tensor[0, 0])

    singleton_rows = ranked_singletons(
        fix_scorer, model, image_tensor, structs_tensor, candidate_bank, targets
    )

    beam = []
    for idx, _, _, _ in singleton_rows[:beam_width]:
        groups = [candidate_bank[idx].copy()]
        metrics = metrics_fix_only(fix_scorer, model, image_tensor, structs_tensor, groups, targets)
        beam.append({"groups": groups, "metrics": metrics, "trace": [f"start:{idx}"]})

    if not beam:
        raise ValueError("Candidate bank is empty.")

    history = []

    for round_idx in range(num_rounds):
        candidates = []
        for state in beam:
            candidates.append(state)
            for _ in range(expansions_per_state):
                action = rng.choice(["modify", "add", "delete", "replace", "merge"])
                proposal = apply_action(
                    state["groups"], action, candidate_bank, image01, rng, max_groups=max_groups
                )
                metrics = metrics_fix_only(
                    fix_scorer, model, image_tensor, structs_tensor, proposal, targets
                )
                candidates.append({
                    "groups": [g.copy() for g in proposal],
                    "metrics": metrics,
                    "trace": state["trace"] + [action],
                })

        candidates.sort(
            key=lambda s: (s["metrics"]["fix"], s["metrics"]["div"]),
            reverse=True,
        )

        next_beam = []
        for cand in candidates:
            keep = True
            for existing in next_beam:
                if len(cand["groups"]) == len(existing["groups"]):
                    mean_iou = np.mean([
                        max(mask_iou(g, h) for h in existing["groups"])
                        for g in cand["groups"]
                    ])
                    if mean_iou > 0.80:
                        keep = False
                        break
            if keep:
                next_beam.append(cand)
            if len(next_beam) >= beam_width:
                break

        beam = next_beam
        history.append({
            "round": round_idx,
            "beam_fix": [s["metrics"]["fix"] for s in beam],
            "beam_div": [s["metrics"]["div"] for s in beam],
        })

    best = beam[0]
    return best["groups"], history, best["metrics"], best["trace"]

## Experiment 3: actual RL-style policy learning

This part **really is RL-like**.

It learns a small policy over action types:
- modify
- add
- delete
- replace
- merge

### Important caveat

This policy only learns **how to edit within the proposal space**.
It does **not** solve the “generate features from scratch” problem.

So if this stage underperforms, the bottleneck is often still the proposal bank.

In [ ]:
ACTION_NAMES = ["modify", "add", "delete", "replace", "merge"]
ACTION_TO_IDX = {a: i for i, a in enumerate(ACTION_NAMES)}


def state_features(current_groups, current_metrics, max_groups=8):
    return np.array([
        current_metrics["fix"],
        current_metrics["div"],
        current_metrics["faith"],
        len(current_groups) / max(max_groups, 1),
        np.mean([g.sum() for g in current_groups]) / (current_groups[0].size if current_groups else 1.0),
    ], dtype=np.float32)


class ActionPolicy(nn.Module):
    def __init__(self, in_dim=5, hidden=32, n_actions=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
def collect_rl_episode(
    policy,
    fix_scorer,
    model,
    image_tensor,
    structs_tensor,
    init_groups,
    candidate_bank,
    targets,
    horizon=20,
    max_groups=8,
    seed=0,
):
    rng = np.random.default_rng(seed)
    image01 = to_numpy01(image_tensor[0, 0])

    current = [g.copy() for g in init_groups]
    current_metrics = metrics_fix_only(fix_scorer, model, image_tensor, structs_tensor, current, targets)

    log_probs = []
    rewards = []
    trajectory = [dict(step=0, action="init", **current_metrics)]

    for step in range(1, horizon + 1):
        s = state_features(current, current_metrics, max_groups=max_groups)
        s_t = torch.from_numpy(s).unsqueeze(0).to(next(policy.parameters()).device)

        logits = policy(s_t)
        dist = torch.distributions.Categorical(logits=logits)
        action_idx = dist.sample()
        action_name = ACTION_NAMES[int(action_idx.item())]

        proposal = apply_action(current, action_name, candidate_bank, image01, rng, max_groups=max_groups)
        proposal_metrics = metrics_fix_only(
            fix_scorer, model, image_tensor, structs_tensor, proposal, targets
        )

        reward = proposal_metrics["fix"] - current_metrics["fix"]

        log_probs.append(dist.log_prob(action_idx))
        rewards.append(float(reward))

        current = [g.copy() for g in proposal]
        current_metrics = dict(proposal_metrics)

        trajectory.append(dict(step=step, action=action_name, reward=reward, **current_metrics))

    return log_probs, rewards, trajectory, current, current_metrics

In [ ]:
def discounted_returns(rewards, gamma=0.95):
    out = []
    running = 0.0
    for r in rewards[::-1]:
        running = float(r) + gamma * running
        out.append(running)
    out = out[::-1]
    out = np.asarray(out, dtype=np.float32)
    if out.std() > 1e-8:
        out = (out - out.mean()) / (out.std() + 1e-8)
    return out


def train_action_policy_reinforce(
    train_indices,
    dataset,
    fix_scorer,
    model,
    layer_name,
    epochs=5,
    episodes_per_epoch=16,
    max_candidates=10,
    init_groups=3,
    horizon=15,
    lr=1e-3,
    gamma=0.95,
    seed=0,
):
    set_seed(seed)
    policy = ActionPolicy().to(device)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)

    history = []

    for epoch in range(epochs):
        epoch_losses = []
        epoch_returns = []

        for _ in tqdm(range(episodes_per_epoch), desc=f"epoch {epoch+1}/{epochs}"):
            idx = int(np.random.choice(train_indices))
            item = dataset[idx]

            image_tensor = item["image"].unsqueeze(0).float()
            gt_pathols = item["pathols"].unsqueeze(0).float()
            structs = item["structs"].float()

            bank = build_candidate_bank(
                model=model,
                image_tensor=image_tensor,
                gt_pathols=gt_pathols,
                layer_name=layer_name,
                max_candidates=max_candidates,
                min_area=120,
            )
            if len(bank["groups"]) == 0:
                continue

            singleton_rows = ranked_singletons(
                fix_scorer, model, image_tensor, structs, bank["groups"], bank["targets"]
            )
            start_groups = seed_init_groups(bank["groups"], singleton_rows, init_groups=init_groups)

            log_probs, rewards, traj, _, _ = collect_rl_episode(
                policy=policy,
                fix_scorer=fix_scorer,
                model=model,
                image_tensor=image_tensor,
                structs_tensor=structs,
                init_groups=start_groups,
                candidate_bank=bank["groups"],
                targets=bank["targets"],
                horizon=horizon,
                seed=int(np.random.randint(0, 10_000_000)),
            )

            if len(log_probs) == 0:
                continue

            returns = discounted_returns(rewards, gamma=gamma)
            returns_t = torch.tensor(returns, device=device)
            log_probs_t = torch.stack(log_probs)

            loss = -(log_probs_t * returns_t).sum()

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            epoch_losses.append(float(loss.item()))
            epoch_returns.append(float(np.sum(rewards)))

        summary = {
            "epoch": epoch + 1,
            "mean_loss": np.mean(epoch_losses) if epoch_losses else np.nan,
            "mean_episode_return": np.mean(epoch_returns) if epoch_returns else np.nan,
        }
        history.append(summary)
        print(summary)

    return policy, pd.DataFrame(history)

In [ ]:
def rollout_learned_policy(
    policy,
    fix_scorer,
    model,
    image_tensor,
    structs_tensor,
    init_groups,
    candidate_bank,
    targets,
    horizon=25,
    max_groups=8,
    seed=0,
):
    rng = np.random.default_rng(seed)
    image01 = to_numpy01(image_tensor[0, 0])

    current = [g.copy() for g in init_groups]
    current_metrics = metrics_fix_only(fix_scorer, model, image_tensor, structs_tensor, current, targets)

    best = [g.copy() for g in current]
    best_metrics = dict(current_metrics)
    history = [dict(step=0, action="init", **current_metrics)]

    for step in range(1, horizon + 1):
        with torch.no_grad():
            s = state_features(current, current_metrics, max_groups=max_groups)
            s_t = torch.from_numpy(s).unsqueeze(0).to(next(policy.parameters()).device)
            logits = policy(s_t)
            action_idx = torch.argmax(logits, dim=-1).item()
            action_name = ACTION_NAMES[int(action_idx)]

        proposal = apply_action(current, action_name, candidate_bank, image01, rng, max_groups=max_groups)
        proposal_metrics = metrics_fix_only(
            fix_scorer, model, image_tensor, structs_tensor, proposal, targets
        )

        current = [g.copy() for g in proposal]
        current_metrics = dict(proposal_metrics)

        if better_by_fix_then_div(current_metrics, best_metrics):
            best = [g.copy() for g in current]
            best_metrics = dict(current_metrics)

        history.append(dict(step=step, action=action_name, **current_metrics))

    return best, history, best_metrics

## Single-image walkthrough

Start here before batch evaluation.

In [ ]:
TEST_INDEX = 0
FIX_STEPS = 80
MAX_CANDIDATES = 12
INIT_GROUPS = 4
MAX_GROUPS = 8

set_seed(0)

item = test_ds[TEST_INDEX]
image_tensor = item["image"].unsqueeze(0).float()
gt_pathols = item["pathols"].unsqueeze(0).float()
structs = item["structs"].float()
structs_np = structs.numpy().astype(np.uint8)

bank = build_candidate_bank(
    model=task_model,
    image_tensor=image_tensor,
    gt_pathols=gt_pathols,
    layer_name=feature_layer,
    max_candidates=MAX_CANDIDATES,
    min_area=120,
)

print("targets:", [PATHOLOGY_NAMES[t] for t in bank["targets"]])
print("candidate count:", len(bank["groups"]))

singleton_rows = ranked_singletons(
    fix_scorer, task_model, image_tensor, structs, bank["groups"], bank["targets"]
)
init_groups = seed_init_groups(bank["groups"], singleton_rows, init_groups=INIT_GROUPS)

init_metrics = metrics_fix_only(
    fix_scorer, task_model, image_tensor, structs, init_groups, bank["targets"]
)
print("initial metrics:", init_metrics)

In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(bank["image01"], cmap="gray", vmin=0, vmax=1)
plt.imshow(bank["cam"], cmap="magma", alpha=0.45)
plt.title("Grad-CAM style seed heatmap")
plt.axis("off")
plt.show()

plot_group_grid(bank["image01"], bank["groups"], "Candidate bank")
plot_group_grid(bank["image01"], init_groups, "Initial FIX-ranked groups")

In [ ]:
best_fix_groups, fix_history, best_fix_metrics = improve_groups_fix_only(
    fix_scorer=fix_scorer,
    model=task_model,
    image_tensor=image_tensor,
    structs_tensor=structs,
    init_groups=init_groups,
    candidate_bank=bank["groups"],
    targets=bank["targets"],
    num_steps=FIX_STEPS,
    max_groups=MAX_GROUPS,
    seed=0,
)

print("best FIX-only metrics:", best_fix_metrics)
plot_group_grid(bank["image01"], best_fix_groups, "Best groups after FIX-only search")
plot_structure_matches(bank["image01"], best_fix_groups, structs_np)

In [ ]:
fix_df = pd.DataFrame(fix_history)
display(fix_df.head())

plt.figure(figsize=(8, 4))
plt.plot(fix_df["step"], fix_df["fix"], label="FIX")
plt.plot(fix_df["step"], fix_df["div"], label="diversity")
plt.plot(fix_df["step"], fix_df["faith"], label="pathology_drop")
plt.xlabel("step")
plt.ylabel("score")
plt.title("FIX-only search trajectory")
plt.legend()
plt.show()

print("FIX delta:", best_fix_metrics["fix"] - init_metrics["fix"])

## Random restarts on the same image

In [ ]:
restart_best_groups, restart_runs, restart_best_metrics, singleton_rows = run_fix_search_with_restarts(
    fix_scorer=fix_scorer,
    model=task_model,
    image_tensor=image_tensor,
    structs_tensor=structs,
    candidate_bank=bank["groups"],
    targets=bank["targets"],
    n_restarts=8,
    num_steps=60,
    init_groups=INIT_GROUPS,
    max_groups=MAX_GROUPS,
    seed=0,
)

restart_table = pd.DataFrame([
    {
        "restart": run["restart"],
        "fix": run["metrics"]["fix"],
        "div": run["metrics"]["div"],
        "faith": run["metrics"]["faith"],
    }
    for run in restart_runs
]).sort_values(["fix", "div"], ascending=False)

display(restart_table)
print("best restart metrics:", restart_best_metrics)
plot_group_grid(bank["image01"], restart_best_groups, "Best groups after random restarts")

## Beam search on the same image

In [ ]:
beam_groups, beam_history, beam_metrics, beam_trace = beam_search_fix(
    fix_scorer=fix_scorer,
    model=task_model,
    image_tensor=image_tensor,
    structs_tensor=structs,
    candidate_bank=bank["groups"],
    targets=bank["targets"],
    beam_width=4,
    expansions_per_state=4,
    num_rounds=10,
    max_groups=MAX_GROUPS,
    seed=0,
)

print("beam metrics:", beam_metrics)
print("beam trace:", beam_trace)
plot_group_grid(bank["image01"], beam_groups, "Best groups after beam search")

## Optional: train the RL-style action policy

This is the part that is actually learned.

It is still deliberately small, because the goal is to answer:
> Does a learned action policy beat plain FIX search?

A good first comparison is:
- FIX-only search
- FIX search + restarts / beam
- learned policy rollout

In [ ]:
# Keep this modest at first.
TRAIN_INDICES = list(range(min(32, len(test_ds))))

policy, rl_train_df = train_action_policy_reinforce(
    train_indices=TRAIN_INDICES,
    dataset=test_ds,
    fix_scorer=fix_scorer,
    model=task_model,
    layer_name=feature_layer,
    epochs=4,
    episodes_per_epoch=12,
    max_candidates=10,
    init_groups=3,
    horizon=12,
    lr=1e-3,
    gamma=0.95,
    seed=0,
)

display(rl_train_df)

In [ ]:
policy_groups, policy_history, policy_metrics = rollout_learned_policy(
    policy=policy,
    fix_scorer=fix_scorer,
    model=task_model,
    image_tensor=image_tensor,
    structs_tensor=structs,
    init_groups=init_groups,
    candidate_bank=bank["groups"],
    targets=bank["targets"],
    horizon=20,
    max_groups=MAX_GROUPS,
    seed=0,
)

print("learned-policy metrics:", policy_metrics)
plot_group_grid(bank["image01"], policy_groups, "Groups from learned action policy")

## Small-batch evaluation

Use this to see what actually helps **on average**.

This is the main table to look at:
- initial FIX
- FIX-only local search
- FIX + restarts
- FIX + beam
- learned policy rollout

In [ ]:
def evaluate_methods_on_subset(
    dataset,
    indices,
    fix_scorer,
    model,
    layer_name,
    fix_steps=60,
    restarts=6,
    beam_rounds=8,
    max_candidates=12,
    init_groups=4,
    max_groups=8,
    policy=None,
):
    rows = []

    for idx in tqdm(indices):
        item = dataset[idx]
        image_tensor = item["image"].unsqueeze(0).float()
        gt_pathols = item["pathols"].unsqueeze(0).float()
        structs = item["structs"].float()

        bank = build_candidate_bank(
            model=model,
            image_tensor=image_tensor,
            gt_pathols=gt_pathols,
            layer_name=layer_name,
            max_candidates=max_candidates,
            min_area=120,
        )
        if len(bank["groups"]) == 0:
            continue

        singleton_rows = ranked_singletons(
            fix_scorer, model, image_tensor, structs, bank["groups"], bank["targets"]
        )
        start_groups = seed_init_groups(bank["groups"], singleton_rows, init_groups=init_groups)
        init_metrics = metrics_fix_only(
            fix_scorer, model, image_tensor, structs, start_groups, bank["targets"]
        )

        fix_groups, _, fix_metrics = improve_groups_fix_only(
            fix_scorer=fix_scorer,
            model=model,
            image_tensor=image_tensor,
            structs_tensor=structs,
            init_groups=start_groups,
            candidate_bank=bank["groups"],
            targets=bank["targets"],
            num_steps=fix_steps,
            max_groups=max_groups,
            seed=idx,
        )

        restart_groups, _, restart_metrics, _ = run_fix_search_with_restarts(
            fix_scorer=fix_scorer,
            model=model,
            image_tensor=image_tensor,
            structs_tensor=structs,
            candidate_bank=bank["groups"],
            targets=bank["targets"],
            n_restarts=restarts,
            num_steps=max(20, fix_steps // 2),
            init_groups=init_groups,
            max_groups=max_groups,
            seed=idx,
        )

        beam_groups, _, beam_metrics, _ = beam_search_fix(
            fix_scorer=fix_scorer,
            model=model,
            image_tensor=image_tensor,
            structs_tensor=structs,
            candidate_bank=bank["groups"],
            targets=bank["targets"],
            beam_width=4,
            expansions_per_state=4,
            num_rounds=beam_rounds,
            max_groups=max_groups,
            seed=idx,
        )

        row = {
            "index": idx,
            "init_fix": init_metrics["fix"],
            "fix_search": fix_metrics["fix"],
            "restart_search": restart_metrics["fix"],
            "beam_search": beam_metrics["fix"],
        }

        if policy is not None:
            policy_groups, _, policy_metrics = rollout_learned_policy(
                policy=policy,
                fix_scorer=fix_scorer,
                model=model,
                image_tensor=image_tensor,
                structs_tensor=structs,
                init_groups=start_groups,
                candidate_bank=bank["groups"],
                targets=bank["targets"],
                horizon=20,
                max_groups=max_groups,
                seed=idx,
            )
            row["learned_policy"] = policy_metrics["fix"]

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
EVAL_INDICES = list(range(min(8, len(test_ds))))

eval_df = evaluate_methods_on_subset(
    dataset=test_ds,
    indices=EVAL_INDICES,
    fix_scorer=fix_scorer,
    model=task_model,
    layer_name=feature_layer,
    fix_steps=50,
    restarts=4,
    beam_rounds=6,
    max_candidates=12,
    init_groups=4,
    max_groups=8,
    policy=policy,
)

display(eval_df)

if len(eval_df):
    summary = pd.DataFrame({
        "mean": eval_df.mean(numeric_only=True),
        "std": eval_df.std(numeric_only=True),
    })
    display(summary)

## How to interpret the results

### If FIX-only search improves over initialization
That means FIX can guide iterative improvement in your setup.

### If restarts or beam help a lot
That means the problem is mostly **search quality**, not “lack of RL”.

### If the learned policy beats plain FIX-only search
Then there is real value in policy learning.

### If nothing helps much
Then the bottleneck is probably:
- weak candidate bank
- poor feature layer
- overly redundant masks
- mutation operators that are too local

In that case, improve the proposal space first.

## Best practical strategy

If your end goal is stronger interpretable groups, the usual best order is:

1. improve candidate proposals  
2. run FIX-only search  
3. add restarts / beam  
4. only then try learned policy optimization

That is usually more productive than starting with “deep RL”.